In [ ]:
# ==============================================================================
# GLINKER — Medical Intake Pipeline
# main.ipynb  (orchestration only — all logic lives in glinker/ and pipeline.py)
#
# Run cells top-to-bottom on first use.
# On subsequent runs, skip Cell 5 (corpus already indexed) unless you want
# to rebuild the knowledge base from scratch.
# ==============================================================================


In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
#
# The repo is cloned into /kaggle/working/curesense-project/ on first run.
# On later runs in the same session it does a git pull to get latest changes.
# REPO_DIR is added to sys.path so glinker/, pipeline.py, and api/ are
# importable immediately after the clone.
#
import sys, os, subprocess

REPO_URL  = 'https://github.com/moizaimran/curesense-project.git'
BRANCH    = 'hassan-branch'
REPO_DIR  = '/kaggle/working/curesense-project'

# If your repo is PRIVATE: add a Kaggle secret named 'GH_TOKEN'
# (Settings -> Secrets -> Add new secret, paste a GitHub Personal Access Token)
# and uncomment the two lines below:
# from kaggle_secrets import UserSecretsClient
# _token   = UserSecretsClient().get_secret('GH_TOKEN')
# REPO_URL = REPO_URL.replace('https://', f'https://{_token}@')

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)


In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
!pip install -r {REPO_DIR}/requirements.txt -q
!pip install torchvision --upgrade --quiet


In [ ]:
# ── Cell 3: Secrets + OpenAI client ──────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
import glinker.config as cfg

_secrets = UserSecretsClient()
ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))
print('OpenAI client ready')


In [ ]:
# ── Cell 4: Load heavy models (Whisper + GLiNER) ──────────────────────────────
import torch
import whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)


In [ ]:
# ── Cell 5: Load or build the RAG index ──────────────────────────────────────
#
# Priority order:
#   1. /kaggle/working/rag_index/        — already built this session, skip
#   2. /kaggle/input/uresense-rag-index/ — pre-built Kaggle dataset attached,
#                                          copy to working dir (seconds, not minutes)
#   3. Build from scratch                — only needed the very first time ever
#
import os, shutil
from glinker.rag.ingestion import build_index
import glinker.config as cfg

WORKING_INDEX = cfg.RAG_INDEX_DIR                    # /kaggle/working/rag_index
DATASET_INDEX = '/kaggle/input/uresense-rag-index'   # pre-built Kaggle dataset

if os.path.exists(f"{WORKING_INDEX}/index.faiss"):
    print("Index already in working dir — skipping.")

elif os.path.exists(f"{DATASET_INDEX}/index.faiss"):
    print("Pre-built dataset found — copying to working dir ...")
    os.makedirs(WORKING_INDEX, exist_ok=True)
    for fname in ("index.faiss", "chunks.json"):
        shutil.copy(f"{DATASET_INDEX}/{fname}", f"{WORKING_INDEX}/{fname}")
    print("Copied. Cell 6 will load it.")

else:
    print("No index found — building from scratch (20-40 min) ...")
    build_index(textbooks_limit=5000, guidelines_limit=2000)
    print("Index built and saved to", WORKING_INDEX)



In [ ]:
# ── Cell 6: Load RAG index into memory ───────────────────────────────────────
from glinker.rag.retrieval import load_index
load_index()


In [ ]:
# ── Cell 7: Load disease ranking datasets (optional) ─────────────────────────
# Requires the 9 Kaggle symptom-disease datasets attached via Add Data.
# The pipeline degrades gracefully (no ranking) if none are attached.
from glinker.disease.ranker import load_datasets
load_datasets()


In [ ]:
# ── Cell 8: Launch Flask API + expose via Ngrok ───────────────────────────────
#
# Starts Flask in a background daemon thread and tunnels it via Ngrok.
# The Ngrok auth token was already set in Cell 3 — no need to pass it again.
#
# Every time this notebook restarts, Ngrok assigns a new URL.
# Copy the printed URL into your Express .env as AI_SERVICE_URL.
#
from api.app import launch

API_URL = launch(port=5001)
print(f"\nExpress .env  →  AI_SERVICE_URL={API_URL}")
